# ISA 444 Final Project

**Overview**

This project focuses on hotel demand forecasting and aims to predict the occupancy rates over the next 28 days for 17 unique hotels in the dataset. These hotels vary in terms of type and location. To forecast, this analysis incorporates a variety of baseline, machine learning, neural, and transformer models including:

*   Naive
*   Seasonal Naive
*   AutoETS
*   AutoARIMA
*   LightGBM
*   NBEATS
*   NHITS
*   Chronos
*   TabPFN

**Baseline Models**

The baseline models simply use only the time series itself to forecast its future values as they cannot take predictors. For the seasonal models (SeasonalNaive), multiple seasonal periods were used including 7, 14, 28, and 365 days.

**Machine Learning, Neural, and Transformer Models**


The machine learning models leverage many of the predictors in the dataset including holiday, target day, target year, etc. However, the on the books 1 - 28 columns were excluded as they caused data leakage issues.

***Disclaimer***

The reasoning for the seasonal periods for the baseline models and the lags for the ML models will be explained in further detail below.

### Step #1 - Import the Libraries

In [ ]:
!pip install timecopilot
!pip install utilsforecast statsforecast mlforecast neuralforecast tabpfn

In [27]:
# Import the neessary libraries we need
import pandas as pd
from utilsforecast.plotting import plot_series
from statsforecast.models import Naive, SeasonalNaive, AutoETS, AutoARIMA
from statsforecast import StatsForecast
from utilsforecast.losses import bias, mae, rmse, mape
from utilsforecast.evaluation import evaluate
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from lightgbm import LGBMRegressor
from neuralforecast import NeuralForecast
from neuralforecast.auto import AutoNBEATS, AutoNHITS
from neuralforecast.models import NBEATS, NHITS
from timecopilot import TimeCopilotForecaster
from timecopilot.models.foundation.chronos import Chronos
from ipywidgets import interact, fixed
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from tabpfn import TabPFNRegressor

### Step #2 - Read and View the Data

In [ ]:
# Next, we are going to read the data
df = pd.read_parquet('sample_hotels.parquet')
df.head() # View the first few rows of the data

In [ ]:
# Now, we are going to use the plot_series function to view the data
plot_series(df, engine = 'matplotlib', max_ids = 19) # Plots all hotel time series with max_ids

**Plot Series Conclusion**

Based on the output of the plot_series function, it is concluded that all time series posses normal none-NA values except hotel_28 and hotel_77. Because of this, these hotels will be removed from the analysis in the next code chunk.

Also, it should be noted that some time series have values that are near 0, which means that MAPE is not a very reliable metric here as it will have inflated / misleading values.

In [41]:
# Based on this, we can see that hotel77 and hotel28 have a lot of missing data, so we will remove them
df = df.query('unique_id not in ["hotel_77", "hotel_28"]')
df = df[df['unique_id'].notna()].reset_index(drop = True)

# Create a baseline dataframe for the baseline models
df_baseline = df[['unique_id', 'ds', 'y']] # Selects only the required columns

In [ ]:
# Next, we are going to plot the ACF and PACF for the models
plot_acf(df['y'], lags = 30);
plot_pacf(df['y'], lags = 30);

**Autocorrelation Function Output**

The following ACF plot was generated using the combined hotel data to examine the overall dependency on past time series. We observe repeating spikes at approximately 7-day intervals (7, 14, 21, and 28) suggesting that weekly seasonality is common across many of the hotel series.

**Partial Autocorrelation Function Output**

The PACF plot shows strong short term dependency at lag 1 with smaller spikes at weekly intervals. After accounting for intermediate effects, the remaining weekly spikes remain outside the confidence bands, which supports the presence of recurring weekly seasonal behavior.

**Conclusions**

Based on the ACF and PACF outputs, the hotel occupancy data displays strong persistence and evidence of weekly seasonality. To account for this, we will evaluate several SeasonalNaive models using seasonal periods of 7, 14, 28, and 365 days to compare forecasting performance across multiple seasonal intervals.

### Step #3 - Fit Baseline Models and Obtain Metrics

In [43]:
# Partition the data into a train / test split for baseline models
train = df_baseline.query('ds < "2023-06-03"') # Ensures there are only 28 days to forecast
test = df_baseline.query('ds >= "2023-06-03"') # Ensures there are only 28 days to forecast

In [44]:
# Create the statsforecast object and the list of models
models = [Naive(), SeasonalNaive(season_length = 7, alias = 'SeasonalNaive_7'), SeasonalNaive(season_length = 14, alias = 'SeasonalNaive_14'),
          SeasonalNaive(season_length = 28, alias = 'SeasonalNaive_28'), SeasonalNaive(season_length = 365, alias = 'SeasonalNaive_365'), AutoETS(), AutoARIMA()]
sf = StatsForecast(models = models, freq = 'D')

# Use cross validation to obtain forecasts that way
baseline_cv_forecasts = sf.cross_validation(h = 28, df = train, n_windows = 5, step_size = 28) # Ensures non-overlapping windows

In [45]:
# Evaluate the forecasts using the evaluate function and the different loss functions we imported
eval_df = evaluate(baseline_cv_forecasts, metrics = [bias, mae, rmse, mape])
eval_df.head() # View the first few rows of the evaluation results

# Group the models by metric and calculate the mean and median
baseline_eval_df = eval_df.groupby(['unique_id', 'metric']).mean()

In [46]:
# Use the test data to predict using all the models
baseline_forecasts = sf.forecast(h = 28, df = test) # Forecast with the test data
baseline_forecasts.to_csv('baseline_forecasts.csv', index = False) # Save the forecasts to a CSV file

### Step #4 - ML, Neural, and TimeCopilot Data Preprocessing

In [47]:
# Encode the categorical variables as integers
df = pd.get_dummies(df, columns = ['holiday_flag', 'target_day', 'target_month', 'location_type', 'hotel_type'])

# Partition the data for ML, neural, and timecopilot models
train_ml = df.query('ds < "2023-06-03"')
test_ml = df.query('ds >= "2023-06-03"')

# Drop the day 1 - 28 otb to prevent data leakage in our modeling
otb_cols = [f'otb_{i}' for i in range(1, 29)]
train_ml = train_ml.drop(columns = otb_cols)
test_ml = test_ml.drop(columns = otb_cols)

**Why drop OTB 1 - 28 from the Test Set?**

The On-the-Books (OTB) variables for days 1 - 28 were removed because they contain information from within the forecast horizon being predicted. Since the objective is to generate a 28-day forecast, these variables would not yet be fully observable at the forecast time. Including them would introduce data leakage, allowing the machine learning models to access information that would only become available closer to the target date. This could inflate model performance and produce unusually accurate forecasts during model evaluation.



### Step #5 - Fit the ML Models and Obtain Metrics

In [ ]:
# Creates the MLForecast object and the list of models
ml_models = [LGBMRegressor()]
mlf = MLForecast(models = ml_models, freq = 'D', lags = range(29, 61))

# Use cross validation to obtain forecasts
ml_cv = mlf.cross_validation(h = 28, df = train_ml, n_windows = 5, step_size = 28, static_features = []) # Ensures non-overlapping windows

**Why exclude OTB lags 1 - 28?**

Hotel occupancy rates are often highly correlated with recent observations. Including lag features within the 1 - 28 day forecast horizon may cause the machine learning models to rely too heavily on short-term rates, where future occupancy is "learned" to be similar to recent occupancy levels. As a result, the models may focus primarily on recent autocorrelation rather than learning broader patterns. To maintain consistency with the 28-day forecasting horizon ensure the models generalize well, occupancy lags within the 1 - 28 day window were excluded.

In [49]:
# Evaluate using the metrics from above
ml_eval_df = evaluate(ml_cv, metrics = [bias, mae, rmse, mape])
ml_eval_df = ml_eval_df.groupby(['unique_id', 'metric']).mean()

In [ ]:
# Use the testing set to predict using the ML model
ml_forecasts = mlf.fit(train_ml, static_features = []) # Fit the model first
ml_forecasts = mlf.predict(h = 28, X_df = test_ml) # Use the predict function as opposed to forecast
ml_forecasts.to_csv('ml_forecasts.csv', index = False) # Save the forecasts

### Step #6 - Fit the Neural Models and Obtain Metrics

In [ ]:
# Create the NeuralForecast object and the list of models
n_models = [NBEATS(h = 28, input_size = 60), NHITS(h = 28, input_size = 60)]
nf = NeuralForecast(models = n_models, freq = 'D')

# Use cross validation to obtain forecasts
n_cv = nf.cross_validation(h = 28, df = train_ml, n_windows = 2, step_size = 28)

In [52]:
# Evaluate the neural models
n_eval_df = evaluate(n_cv, metrics = [bias, mae, rmse, mape])
n_eval_df = n_eval_df.groupby(['unique_id', 'metric']).mean()

In [ ]:
# Write the final forecasting results to csv
n_forecasts = nf.fit(train_ml)
n_forecasts = nf.predict()
n_forecasts.to_csv('n_forecasts.csv', index = False)

### Step #7 - Fit the TimeCopilot Models and Obtain Metrics

In [ ]:
# Define the time copilot forecasting object
tcf = TimeCopilotForecaster(models = [Chronos(repo_id="amazon/chronos-bolt-small")])

# Perform cross validation with the Chronos model
tc_cv = tcf.cross_validation(df = train, h = 28, freq = 'D', n_windows = 5, step_size = 28)

In [19]:
# Evaluate the performanc of the Chronos model
tc_eval_df = evaluate(tc_cv, metrics = [bias, mae, rmse, mape])
tc_eval_df = tc_eval_df.groupby(['unique_id', 'metric']).mean()

In [ ]:
# Write the final results of the forecast to CSV
tc_forecasts = tcf.forecast(h = 28, df = test)
tc_forecasts.to_csv('TC_Forecasts.csv', index = False)

### Step #8 - Count Model Wins and View Forecasts

In [55]:
# Merge all the evaluation data frames into one df
final_eval_df = pd.concat([baseline_eval_df, ml_eval_df.drop(columns = 'cutoff'), tc_eval_df.drop(columns = 'cutoff'),
                           n_eval_df.drop(columns = 'cutoff')], axis = 1)
final_eval_df = final_eval_df.reset_index()

In [56]:
# Define the model columns
model_columns = ['Naive', 'SeasonalNaive_7', 'SeasonalNaive_14', 'SeasonalNaive_28', 'SeasonalNaive_365', 'AutoETS', 'AutoARIMA',
                             'LGBMRegressor', 'NBEATS', 'NHITS', 'Chronos']

# Aggregate across all metrics to find the best model per series
hotel_avg = (final_eval_df.groupby(['unique_id'])[model_columns].mean())
hotel_avg['best_model_per_series'] = hotel_avg.idxmin(axis = 1)

# Write the model wins to CSV
hotel_avg.to_csv('Hotel_Series_Model_Winners.csv', index = False)

In [ ]:
# Merge all the forecasts together
forecasts_all = pd.concat([baseline_forecasts, ml_forecasts.drop(columns = ['unique_id', 'ds']),
                           tc_forecasts.drop(columns = ['unique_id', 'ds']), n_forecasts.drop(columns = ['unique_id', 'ds'])], axis = 1)

# Write a function to visualize the forecasts per model
def view_forecasts(model_name):

  # Define the figsize
  plt.figure(figsize = (12, 6))

  # Plot the time series with the model plot_series(df, model_name)
  fig = plot_series(df = df, forecasts_df = forecasts_all, models = [model_name], max_ids = 17)
  plt.show()
  return fig

# Make an interactive dropdown window
interact(view_forecasts, model_name = ['Naive', 'SeasonalNaive_7', 'SeasonalNaive_14', 'SeasonalNaive_28', 'SeasonalNaive_365', 'AutoETS', 'AutoARIMA',
                             'LGBMRegressor', 'NBEATS', 'NHITS', 'Chronos'])

# **For More Information**

For more information on the specific findings and rationals, please view the readme file which contains more in-depth analysis.